# logit-thermo A1 確認（PQC7）：裾の型と、壁の圧力

- **必要な secret：なし**（Drive をつなぐ許可だけ聞かれます）
- **ランタイム：CPU（標準）でOK**。GPU・ハイメモリはいりません。見込みは 10〜20 分くらい。
- 何をするか：ゲート `docs/A1_WALL_GATE.md` v0.3 の手順 3。`PROJECT_A/_LOGITS/PARENT/PQC7` の logit（1,536 行）を sha256 で確かめてから、行ごとに裾の傾き・目印 g と c・壁の圧縮因子・比熱のピークを計算し、J1・J2（裾の型）と R1〜R5（記録）をまとめます。
- 計算のコードは logit-thermo の `scripts/a1_core.py`（sha256 `59716b354921e24e…`）をそのまま埋め込んであります。
- 結果の置き場所：`MyDrive/logit-thermo/A1/PQC7/`（まとめ `a1_summary.json`、行ごとの表 `a1_rows.csv`）。
- **通知**：ntfy の `YOUR_NTFY_TOPIC` に、🚀 始まったとき・🎉 終わったとき・⚠️ エラーで止まったとき（エラーの文面まるごと）に届きます。名札は「logit-thermo A1 確認（PQC7）裾の型と壁の圧力」。ランタイムごと切れたときは届かないので、そのときは Colab を見てください。
- **終わったら自動でランタイムを切ります**（Drive への保存を確定してから）。エラーで止まったときは切らずに残します。切りたくないときは、最初に `AUTO_DISCONNECT = False` を実行してください。
- 使い方：「ランタイム」→「すべてのセルを実行」。


In [ ]:
# セル1：通知のしくみ（ntfy）と、Drive をつなぐ
import urllib.request, time
NTFY_TOPIC = 'YOUR_NTFY_TOPIC'
NB_LABEL = 'logit-thermo A1 確認（PQC7）裾の型と壁の圧力'   # どのノートブックからの通知かの名札
def notify(msg, tags='pig'):
    try:
        req = urllib.request.Request('https://ntfy.sh/' + NTFY_TOPIC, data=(msg + '\n[' + NB_LABEL + ']').encode('utf-8'),
                                     headers={'Title': 'logit-thermo - A1 confirm PQC7', 'Tags': tags})
        urllib.request.urlopen(req, timeout=10).read()
        print('通知を送りました：', msg)
    except Exception as e:
        print('通知は送れませんでした（実験には影響しません）：', type(e).__name__)
_NOTIFIED = {'error': False}
def _on_error(result):
    if result.error_in_exec is not None and not _NOTIFIED['error']:
        _NOTIFIED['error'] = True
        import traceback
        e = result.error_in_exec
        txt = ''.join(traceback.format_exception(type(e), e, e.__traceback__))
        notify('⚠️ 止まりました（エラー）：\n' + txt[-3500:], tags='warning')
if not globals().get('_HOOK_ON'):
    get_ipython().events.register('post_run_cell', _on_error); _HOOK_ON = True
notify('🚀 スタートしました（これが届けば通知の設定は OK です）', tags='rocket')

from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# セル2：計算のコード（logit-thermo の scripts/a1_core.py をそのまま埋め込み）
"""A1（ゲート v0.3）の計算。ノートブックにはこのファイルの中身をそのまま埋め込む。

1 行（1 位置の logit、使う番号だけ）ごとに row_metrics() で数字を出し、
summarize() で J1・J2（裾の型の判定）と R1〜R5（記録）をまとめる。
すべて float64。温度は T = 1 の logit を T で割る。
"""
import numpy as np

K = 1000                 # 壁の位置
H = 100                  # 頭として除く順位（1..H-1 を除く）
WIN = (100, 10_000)      # a_win の窓
LOC = (300, 3_000)       # a_loc の窓
S_WIN = (100, 1_001)     # R1 の s の窓
C_SHALLOW, C_DEEP = (100, 1_000), (1_000, 10_000)
GAUSS_K = 0.36           # ガウス型での a_loc ÷ σ（§2.5）
T_GRID = np.exp(np.linspace(np.log(0.2), np.log(8.0), 121))
G_BANDS = ((0.85, 1.15),)   # J1：[0.85, 1.15] がガウス型
C_BANDS = ((0.9, 1.15),)    # J2：[0.9, 1.15] が Zipf 型
BOOT_N, BOOT_SEED = 2000, 20261027


def thresholds(V):
    """p ∝ 1/r（語彙 V）での Zc_full・Zc_tail の値。"""
    w = 1.0 / np.arange(1, V + 1, dtype=np.float64)
    return K * np.log1p(w[K] / w[:K].sum()), K * np.log1p(w[K] / w[H - 1:K].sum())


def _slope(zs, lo, hi):
    r = np.arange(lo, hi + 1)
    return -np.polyfit(np.log(r), zs[r - 1], 1)[0]


def _t_on(zc, theta):
    """(T_on, 端の印)。端の印は None・'low'（最低温ですでに超えている）・'high'（最高温でも届かない）。"""
    above = np.nonzero(zc >= theta)[0]
    if len(above) == 0:
        return np.nan, 'high'
    i = above[0]
    if i == 0:
        return np.nan, 'low'
    t0, t1 = np.log(T_GRID[i - 1]), np.log(T_GRID[i])
    f = (theta - zc[i - 1]) / (zc[i] - zc[i - 1])
    return float(np.exp(t0 + f * (t1 - t0))), None


def _logsumexp(x):
    m = x.max()
    return m + np.log(np.exp(x - m).sum())


def row_metrics(z, theta_full, theta_tail):
    """z：1 行の logit（使う番号だけ、float64）。"""
    zs = np.sort(z)[::-1]
    out = {}
    # 傾き
    r = np.arange(WIN[0], WIN[1] + 1)
    x, y = np.log(r), zs[r - 1]
    sl, ic = np.polyfit(x, y, 1)
    resid = y - (sl * x + ic)
    out['a_win'] = -sl
    out['r2_win'] = 1 - resid.var() / y.var()
    out['a_loc'] = _slope(zs, *LOC)
    out['s'] = _slope(zs, *S_WIN) / out['a_loc']
    out['c'] = _slope(zs, *C_DEEP) / _slope(zs, *C_SHALLOW)
    q25, q75 = np.percentile(z, [25, 75])
    out['sigma_rob'] = (q75 - q25) / 1.349
    out['g'] = out['a_loc'] / (GAUSS_K * out['sigma_rob'])
    # 頭の強さ（T = 1）
    out['h'] = (_logsumexp(zs[:H - 1]) - _logsumexp(zs[H - 1:])) / np.log(10)
    # 圧縮因子（上位 K+1 個だけ使う）
    top = zs[:K + 1]
    zf, zt = [], []
    for T in T_GRID:
        w = np.exp((top - top[0]) / T)
        zf.append(K * np.log1p(w[K] / w[:K].sum()))
        zt.append(K * np.log1p(w[K] / w[H - 1:K].sum()))
    out['t_on_full'], out['edge_full'] = _t_on(np.array(zf), theta_full)
    out['t_on_tail'], out['edge_tail'] = _t_on(np.array(zt), theta_tail)
    # 比熱 C(T) = Var(E) / T²、E = −z
    cs = []
    zc0 = zs - zs[0]
    for T in T_GRID:
        w = np.exp(zc0 / T)
        p = w / w.sum()
        m1 = (p * zs).sum()
        cs.append(((p * zs * zs).sum() - m1 * m1) / (T * T))
    i = int(np.argmax(cs))
    out['t_c'] = float(T_GRID[i]) if 0 < i < len(T_GRID) - 1 else np.nan
    out['edge_c'] = None if 0 < i < len(T_GRID) - 1 else ('low' if i == 0 else 'high')
    return out


def _band(v, lo, hi, names):
    return names[0] if v < lo else (names[1] if v <= hi else names[2])


G_NAMES = ('ガウスより緩い', 'ガウス型', 'Zipf寄り')
C_NAMES = ('浅いほうが急', 'Zipf型', 'ガウス型')


def _spearman(a, b):
    from scipy.stats import spearmanr
    a, b = np.asarray(a, float), np.asarray(b, float)
    ok = np.isfinite(a) & np.isfinite(b)
    if ok.sum() < 3:
        return None
    return float(spearmanr(a[ok], b[ok]).correlation)


def _q(v, p):
    v = np.asarray(v, float)
    v = v[np.isfinite(v)]
    return float(np.percentile(v, p)) if len(v) else None


def _dist(v):
    return {k: _q(v, p) for k, p in (('p10', 10), ('p25', 25), ('median', 50), ('p75', 75), ('p90', 90))}


def _judge(values, prompts, lo, hi, names):
    values, prompts = np.asarray(values, float), np.asarray(prompts)
    med = float(np.median(values))
    ids = np.unique(prompts)
    groups = [values[prompts == i] for i in ids]
    rng = np.random.default_rng(BOOT_SEED)
    meds = [np.median(np.concatenate([groups[j] for j in rng.integers(0, len(groups), len(groups))]))
            for _ in range(BOOT_N)]
    ci = (float(np.percentile(meds, 2.5)), float(np.percentile(meds, 97.5)))
    bands = sorted({_band(ci[0], lo, hi, names), _band(med, lo, hi, names), _band(ci[1], lo, hi, names)},
                   key=names.index)
    share = {n: float(np.mean([_band(v, lo, hi, names) == n for v in values])) for n in names}
    return {'median': med, 'ci95_prompt_boot': ci, 'band_of_median': _band(med, lo, hi, names),
            'clear': len(bands) == 1, 'bands_touched_by_ci': bands, 'share_rows': share}


def summarize(rows, prompts):
    """rows：row_metrics の結果のリスト、prompts：各行のプロンプトの番号。"""
    col = lambda k: np.array([r[k] if r[k] is not None else np.nan for r in rows], float)
    edge = lambda k: [r[k] for r in rows]
    g, c = col('g'), col('c')
    a_loc, a_win = col('a_loc'), col('a_win')
    ton_t, ton_f, s, h, tc = col('t_on_tail'), col('t_on_full'), col('s'), col('h'), col('t_c')
    ratio_t, ratio_f = ton_t / a_loc, ton_f / a_win
    n = len(rows)
    frac = lambda k, v: float(sum(e == v for e in edge(k)) / n)
    return {
        'rows': n, 'prompts': int(len(np.unique(prompts))),
        'J1_g': _judge(g, prompts, *G_BANDS[0], G_NAMES),
        'J2_c': _judge(c, prompts, *C_BANDS[0], C_NAMES),
        'R1': {'ratio_tail': _dist(ratio_t), 's': _dist(s),
               'median_ratio_minus_s': _q(ratio_t - s, 50), 'spearman_ratio_s': _spearman(ratio_t, s),
               'spearman_ratio_c': _spearman(ratio_t, c),
               'v02_H1_check': {'median_ratio': _q(ratio_t, 50), 'spearman_a_loc_t_on_tail': _spearman(a_loc, ton_t)}},
        'R2': {'spearman_h_ratio_full': _spearman(h, ratio_f), 'ratio_full': _dist(ratio_f), 'h': _dist(h)},
        'R3': {'spearman_t_c_t_on_tail': _spearman(tc, ton_t), 'median_t_c_over_t_on_tail': _q(tc / ton_t, 50),
               't_c': _dist(tc)},
        'R4': {'a_win': _dist(a_win), 'a_loc': _dist(a_loc), 'r2_win': _dist(col('r2_win')),
               'sigma_rob': _dist(col('sigma_rob'))},
        'R5': {'tail_low': frac('edge_tail', 'low'), 'tail_high': frac('edge_tail', 'high'),
               'full_low': frac('edge_full', 'low'), 'full_high': frac('edge_full', 'high'),
               'c_low': frac('edge_c', 'low'), 'c_high': frac('edge_c', 'high')},
    }


In [ ]:
# セル3：sha256 を確かめてから、1,536 行を計算してまとめる
import hashlib, io, json, os, platform
MODEL = 'PQC7'
BASE = '/content/drive/MyDrive/PROJECT_A/_LOGITS/PARENT'
OUT = '/content/drive/MyDrive/logit-thermo/A1/PQC7'
os.makedirs(OUT, exist_ok=True)
d = f'{BASE}/{MODEL}/logits'
meta = json.load(open(f'{d}/metadata.json'))
tok_len = int(meta['tokenizer_len'])
assert tok_len == 151665, tok_len
theta_full, theta_tail = thresholds(tok_len)
assert abs(theta_full - 0.133450) < 1e-6 and abs(theta_tail - 0.432732) < 1e-6, (theta_full, theta_tail)
items = sorted(meta['precisions']['fp32']['files'].items(), key=lambda kv: kv[1]['file'])
rows, prompts, ids = [], [], []
t0 = time.time()
for pi, (sid, info) in enumerate(items):
    raw = open(f"{d}/{info['file']}", 'rb').read()
    assert hashlib.sha256(raw).hexdigest() == info['sha256'], f"sha256 が合いません：{info['file']}"
    a = np.load(io.BytesIO(raw)).astype(np.float64)
    assert a.shape == (info['rows'], info['vocab']), a.shape
    for r in range(a.shape[0]):
        rows.append(row_metrics(a[r, :tok_len], theta_full, theta_tail)); prompts.append(pi); ids.append((info['file'], r))
    del a, raw
    print(f"{pi + 1}/{len(items)} ファイルおわり（{time.time() - t0:.0f} 秒）")
summary = summarize(rows, np.array(prompts))
summary.update({'model': MODEL, 'model_id': meta.get('model_id'), 'gate': 'docs/A1_WALL_GATE.md v0.3',
                'a1_core_sha256': '59716b354921e24e22110121d10f3458541bc6a8b3db27129750dcfcccf0bbab', 'files_sha256_ok': len(items), 'numpy': np.__version__,
                'python': platform.python_version(), 'elapsed_seconds': round(time.time() - t0, 1)})
keys = ['a_win', 'r2_win', 'a_loc', 's', 'c', 'sigma_rob', 'g', 'h', 't_on_full', 'edge_full', 't_on_tail', 'edge_tail', 't_c', 'edge_c']
with open(f'{OUT}/a1_rows.csv', 'w') as f:
    f.write('file,row,prompt,' + ','.join(keys) + '\n')
    for (fn, r), p, m in zip(ids, prompts, rows):
        vals = [('' if m[k] is None else (f'{m[k]:.6f}' if isinstance(m[k], float) else str(m[k]))) for k in keys]
        f.write(f'{fn},{r},{p},' + ','.join(vals) + '\n')
summary['rows_csv_sha256'] = hashlib.sha256(open(f'{OUT}/a1_rows.csv', 'rb').read()).hexdigest()
json.dump(summary, open(f'{OUT}/a1_summary.json', 'w'), ensure_ascii=False, indent=1)
print('まとめの大きさ：', os.path.getsize(f'{OUT}/a1_summary.json'), 'バイト')
print(json.dumps(summary, ensure_ascii=False, indent=1))


In [ ]:
# セル4：終わりの通知と、ランタイムを切る
notify('🎉 終わりました：' + str(summary['rows']) + ' 行', tags='tada')
if globals().get('AUTO_DISCONNECT', True):
    print('Drive への書き込みを確定してから、ランタイムを切ります（CU の節約。止めたいときは AUTO_DISCONNECT = False）')
    from google.colab import drive as _drive, runtime as _runtime
    _drive.flush_and_unmount()
    time.sleep(3)
    _runtime.unassign()
